# Signal and Noise Simulator (with Audio)
> *Based on Shannon's Information Theory (1948)*

This notebook simulates how a signal travels through a noisy channel and what happens when you try to recover it.

**New in this version:** You can now *hear* the difference between a clean signal, a corrupted one, and the recovered version not just see it.



### How to use
1. Run **Cell 1** to install and import everything
2. Run **Cell 2** to define all the functions
3. Run **Cell 3** to launch the interactive simulator
4. Adjust the sliders and click **Play Audio** buttons to hear each signal

In [4]:
# Cell 1

# Install and Import needed libraries
# soundfile lets us write audio to a .wav file
# IPython.display.Audio lets Colab play it in the notebook
# Everything else is already in Colab

!pip install soundfile --quiet

import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import ipywidgets as widgets

from scipy import signal
from IPython.display import display, Audio, Markdown, clear_output

print("libraries loaded!")

libraries loaded!


In [5]:
# Cell 2

# For sound to be audible to human ears, we need a freq in the audible range (20Hz–20,000Hz)
# We use 440Hz, the musical note A4 the reference freq to calibrate acoustic instruments, string instruments, pianos etc., for the audio freq
# The original 5Hz sine wave is below human hearing so it would be silent but its kept for plotting so the wave is easy to see

SAMPLE_RATE  = 44100   # Standard audio sample rate (CD quality)
PLOT_FREQ    = 5
AUDIO_FREQ   = 440
DURATION     = 1.0

# Time axis for plotted graphs
t_plot = np.linspace(0, DURATION, 500, endpoint=False)

# Time axis for audio
t_audio = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION), endpoint=False)

# Generate the two versions of the clean signal
clean_plot  = np.sin(2 * np.pi * PLOT_FREQ  * t_plot)   # for graphs
clean_audio = np.sin(2 * np.pi * AUDIO_FREQ * t_audio)  # for ears


# Added Gaussian (random) noise to the signal at a target SNR (Signal to Noise Ratio) level in dB
# High dB = signal much stronger than noise = mostly clear
# Low dB  = noise nearly as strong as signal = very corrupted

def add_noise(sig, snr_db):
    power       = np.mean(sig ** 2)
    noise_power = power / (10 ** (snr_db / 10))
    noise       = np.random.normal(0, np.sqrt(noise_power), len(sig))
    return sig + noise

# A Butterworth low-pass filter is used to filter the noise
# It keeps frequencies below the cutoff (in this case 5Hz for plotting or 440Hz for audio) and removes frequencies above it
# filtfilt applies the filter forwards and backwards so it does not shift your signal in time

def recover_signal(noisy, cutoff, fs, order=4):
    nyquist    = fs / 2
    normalized = cutoff / nyquist
    # Cutoff must be between 0 and 1 when normalised
    normalized = np.clip(normalized, 0.001, 0.999)
    b, a       = signal.butter(order, normalized, btype='low')
    return signal.filtfilt(b, a, noisy)


# For SNR measurement, we subtract the recovered signal from the original to find the residual error
# Then compare how much power is in the signal against the error
# Result is in dB (higher dB is better)

def measure_snr(original, estimate):
    residual     = original - estimate
    signal_power = np.mean(original ** 2)
    noise_power  = np.mean(residual ** 2)
    if noise_power == 0:
        return float('inf')
    return 10 * np.log10(signal_power / noise_power)


# Maps the SNR number to a scenario a Nigerian would recognise
def get_scenario(snr_db):
    if snr_db >= 18:
        return "Clear link — low traffic, strong signal (when most people are asleep)"
    elif snr_db >= 8:
        return "Moderate interference — congested channel, peak hours (usually when people close from work)"
    else:
        return "Heavy degradation — severe noise (rainstorm)"


# A clean 5Hz signal = one spike at 5Hz
# A noisy signal = that spike PLUS random energy spread across all frequencies
# After filtering = the spike remains, the spread is reduced
# This explains visually why the filter works and why it fails at very low SNR

def plot_spectrum(ax, sig, fs, title, color='steelblue'):
    freqs    = np.fft.rfftfreq(len(sig), d=1/fs)
    spectrum = np.abs(np.fft.rfft(sig))
    ax.plot(freqs, spectrum, color=color, linewidth=1.2)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Magnitude")
    ax.grid(alpha=0.3)


# Scales any signal to the range [-1, 1] so it plays at a safe, consistent volume so it would not be "painfully" loud or silent
def normalize_audio(sig):
    peak = np.max(np.abs(sig))
    if peak == 0:
        return sig
    return sig / peak * 0.8   # 0.8 leaves a little headroom


print("All functions defined")
print(f"   Plot signal : {PLOT_FREQ}Hz (visible in graphs)")
print(f"   Audio signal: {AUDIO_FREQ}Hz (audible to human ears)")
print(f"   Sample rate : {SAMPLE_RATE} Hz (CD quality)")

All functions defined
   Plot signal : 5Hz (visible in graphs)
   Audio signal: 440Hz (audible to human ears)
   Sample rate : 44100 Hz (CD quality)


In [7]:
# Cell 3

output_area = widgets.Output()


def run_simulation(snr_db, cutoff_plot, show_spectrum, show_error):
    """Main simulation — runs every time a slider moves."""

    with output_area:
        clear_output(wait=True)

        # Generate all six signals (plot + audio versions)
        # Plot versions (5Hz)
        noisy_plot     = add_noise(clean_plot, snr_db)
        recovered_plot = recover_signal(noisy_plot, cutoff=cutoff_plot, fs=500)

        # Audio versions (440Hz)
        audio_cutoff   = cutoff_plot * (AUDIO_FREQ / PLOT_FREQ)
        audio_cutoff   = min(audio_cutoff, SAMPLE_RATE / 2 - 100)  # safety cap
        noisy_audio     = add_noise(clean_audio, snr_db)
        recovered_audio = recover_signal(noisy_audio, cutoff=audio_cutoff, fs=SAMPLE_RATE)

        # Measure quality or improvement of the SNR
        recovered_snr = measure_snr(clean_plot, recovered_plot)
        improvement   = recovered_snr - snr_db

        # Summary card
        display(Markdown(f"""
---
## Simulation Summary

| | |
|---|---|
| **Scenario** | {get_scenario(snr_db)} |
| **Input SNR** | {snr_db:.1f} dB |
| **Recovered SNR** | {recovered_snr:.2f} dB |
| **Filter improvement** | +{improvement:.2f} dB |
| **Filter cutoff** | {cutoff_plot:.0f} Hz (plot) · {audio_cutoff:.0f} Hz (audio) |

> {'Heavy noise — filter hitting Shannon\'s Channel Capacity limit. No engineering can fully recover this.' if snr_db < 8 else 'Filter recovered the signal successfully.'}
---
"""))

        # Time-domain plots
        fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
        plt.suptitle('Time-Domain View — What the signal looks like', fontsize=13, fontweight='bold')

        # Clean
        axes[0].plot(t_plot, clean_plot, color='#1D9E75', linewidth=2, label='Clean signal')
        axes[0].set_title('1. Original Clean Signal')
        axes[2].set_xlabel('Time (seconds)')
        axes[0].set_ylabel('Amplitude')
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        # Noisy
        axes[1].plot(t_plot, clean_plot, '--', color='#1D9E75', linewidth=1.5, label='Original')
        axes[1].plot(t_plot, noisy_plot, color='#E24B4A', alpha=0.8, linewidth=1, label=f'Noisy ({snr_db:.1f} dB)')
        if show_error:
            axes[1].fill_between(t_plot, clean_plot, noisy_plot, alpha=0.15, color='#E24B4A', label='Noise error')
        axes[1].set_title('2. Signal After Channel Noise')
        axes[2].set_xlabel('Time (seconds)')
        axes[1].set_ylabel('Amplitude')
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        # Recovered
        axes[2].plot(t_plot, clean_plot, '--', color='#1D9E75', linewidth=1.5, label='Original')
        axes[2].plot(t_plot, noisy_plot, color='#E24B4A', alpha=0.3, linewidth=0.8, label='Noisy')
        axes[2].plot(t_plot, recovered_plot, color='#534AB7', linewidth=2, label='Recovered')
        if show_error:
            axes[2].fill_between(t_plot, clean_plot, recovered_plot, alpha=0.15, color='#534AB7', label='Residual error')
        axes[2].set_title('3. Signal After Filter Recovery')
        axes[2].set_xlabel('Time (seconds)')
        axes[2].set_ylabel('Amplitude')
        axes[2].legend()
        axes[2].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

        # Frequency-domain plots
        if show_spectrum:
            fig2, axes2 = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
            plt.suptitle('Frequency-Domain View — WHERE the signal energy lives', fontsize=13, fontweight='bold')
            plot_spectrum(axes2[0], clean_plot,     500, '1. Spectrum: Clean Signal',     '#1D9E75')
            plot_spectrum(axes2[1], noisy_plot,     500, '2. Spectrum: Noisy Signal',     '#E24B4A')
            plot_spectrum(axes2[2], recovered_plot, 500, '3. Spectrum: Recovered Signal', '#534AB7')
            axes2[2].set_xlim(0, 60)
            plt.tight_layout()
            plt.show()

        # Audio section
        display(Markdown("""
---
## Audio Playback

The signal has been shifted to **440Hz** (the musical note A4) so it falls within human hearing range.
You are hearing exactly what you're seeing in the plots above.

| What you will hear | Description |
|---|---|
| **Clean** | The signal from the source (e.g The person initiating the call) |
| **Noisy** | The channel corrupting the signal |
| **Recovered** | The filters attempt to pull the tone back out |

"""))

        # Normalise all three for safe playback
        audio_clean     = normalize_audio(clean_audio)
        audio_noisy     = normalize_audio(noisy_audio)
        audio_recovered = normalize_audio(recovered_audio)

        # Display three labelled audio players
        display(Markdown('**1. Clean signal** — the ideal transmission'))
        display(Audio(audio_clean, rate=SAMPLE_RATE, autoplay=False))

        display(Markdown(f'**2. Noisy signal** — after passing through the channel at {snr_db:.1f} dB SNR'))
        display(Audio(audio_noisy, rate=SAMPLE_RATE, autoplay=False))

        display(Markdown(f'**3. Recovered signal** — after filter (cutoff: {audio_cutoff:.0f} Hz)'))
        display(Audio(audio_recovered, rate=SAMPLE_RATE, autoplay=False))

        display(Markdown('---'))


# Widgets
snr_slider = widgets.FloatSlider(
    value=10, min=0, max=30, step=1,
    description='SNR (dB):',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='480px'),
    continuous_update=False
)

cutoff_slider = widgets.FloatSlider(
    value=20, min=5, max=100, step=1,
    description='Cutoff (Hz):',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='480px'),
    continuous_update=False
)

spectrum_toggle = widgets.Checkbox(value=True,  description='Show frequency spectrum')
error_toggle    = widgets.Checkbox(value=True,  description='Show error shading')

snr_label    = widgets.HTML('<b>SNR</b> — drag left for more noise, right for cleaner signal')
cutoff_label = widgets.HTML('<b>Filter cutoff</b> — drag right to let more frequencies through')

ui = widgets.VBox([
    widgets.HTML('<h1 style="margin-bottom:4px">Signal and Noise Simulator</h1>'),
    widgets.HTML('<p style="color:gray;margin-top:0">Adjust sliders to change channel conditions. Audio players appear below the plots.</p>'),
    snr_label,
    snr_slider,
    cutoff_label,
    cutoff_slider,
    widgets.HBox([spectrum_toggle, error_toggle])
])

interactive = widgets.interactive_output(
    run_simulation,
    {'snr_db': snr_slider, 'cutoff_plot': cutoff_slider,
     'show_spectrum': spectrum_toggle, 'show_error': error_toggle}
)

display(ui, output_area)
interactive.observe(lambda change: run_simulation(
    snr_slider.value, cutoff_slider.value,
    spectrum_toggle.value, error_toggle.value
), names='value')

# Run once on load
run_simulation(snr_slider.value, cutoff_slider.value,
               spectrum_toggle.value, error_toggle.value)

Output()